# Phase 5 — Machine Learning Models

## Goal
Train and compare 3 ML models to predict financial distress.
Find the best model to power our early warning system.

## Input
data/processed/companies_ml_ready.csv
60 companies, 10 ratio features, binary label (0=Healthy, 1=Distressed)

## Models We Will Train
1. Logistic Regression — simple baseline
2. Random Forest — powerful ensemble model
3. XGBoost — state of the art for tabular data

## Success Criteria
- Accuracy > 80%
- High Recall — we must catch as many distressed companies as possible
- ROC AUC > 0.85 — strong overall model quality

## Output
- Best model saved to models/ folder
- Ready for Streamlit dashboard in Phase 7

In [4]:
# Cell 1 — Import libraries and load ML ready dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries — all from scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
import xgboost as xgb
import joblib  # for saving our trained model

# Load ML ready dataset
df = pd.read_csv('../data/processed/companies_ml_ready.csv')

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nHealthy companies: {len(df[df['status'] == 'Healthy'])}")
print(f"Distressed companies: {len(df[df['status'] == 'Distressed'])}")
print(f"\nFeatures available:")
features = [col for col in df.columns 
            if col not in ['company', 'status', 'label']]
for f in features:
    print(f"  - {f}")

Data loaded successfully!
Shape: (60, 13)

Healthy companies: 30
Distressed companies: 30

Features available:
  - debt_to_assets
  - debt_to_equity
  - net_income_margin
  - operating_margin
  - operating_cashflow_ratio
  - cashflow_to_debt
  - asset_turnover
  - interest_burden
  - retained_earnings_ratio
  - cash_flow_margin


In [5]:
# Cell 2 — Prepare data for ML modeling
# Separate features from target, scale, and set up cross validation

# Define our 10 ratio features
features = [
    'debt_to_assets', 'debt_to_equity',
    'net_income_margin', 'operating_margin',
    'operating_cashflow_ratio', 'cashflow_to_debt',
    'asset_turnover', 'interest_burden',
    'retained_earnings_ratio', 'cash_flow_margin'
]

# X = features (what model learns from)
# y = label (what model predicts: 0=Healthy, 1=Distressed)
X = df[features].values
y = df['label'].values

print(f"X shape: {X.shape} → 60 companies, 10 features")
print(f"y shape: {y.shape} → 60 labels")
print(f"\nClass distribution:")
print(f"Healthy (0):    {sum(y == 0)} companies")
print(f"Distressed (1): {sum(y == 1)} companies")

# Scale features using StandardScaler
# Transforms all features to same scale: mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nAfter scaling:")
print(f"Mean of each feature (should be ~0):")
print(X_scaled.mean(axis=0).round(10))

# Set up Stratified 5-Fold Cross Validation
# Stratified = maintains 50/50 balance in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nCross validation setup:")
print(f"5 folds → each fold trains on 48, tests on 12 companies")
print(f"\nData preparation complete! Ready for modeling.")

X shape: (60, 10) → 60 companies, 10 features
y shape: (60,) → 60 labels

Class distribution:
Healthy (0):    30 companies
Distressed (1): 30 companies

After scaling:
Mean of each feature (should be ~0):
[ 0.  0. -0. -0.  0. -0.  0. -0.  0. -0.]

Cross validation setup:
5 folds → each fold trains on 48, tests on 12 companies

Data preparation complete! Ready for modeling.


## Model 1 — Logistic Regression (Baseline)

### Why We Start With Logistic Regression
Every professional ML project starts with a simple baseline.
If our complex models cannot beat the baseline significantly,
it means our features or data need more work — not the model.

Logistic Regression finds the best linear combination of our
10 ratio features to separate healthy from distressed companies.

### Our Minimum Bar
A model that always guesses "Healthy" gets 50% accuracy.
Our logistic regression must beat 50% to be useful.
Our target is 80%+ accuracy.

### Evaluation Metrics We Will Use
- Accuracy: what % of companies did we classify correctly?
- Precision: when we predict distressed, how often are we right?
- Recall: of all truly distressed companies, how many did we catch?
- F1 Score: balance between precision and recall
- ROC AUC: overall model quality from 0.5 (random) to 1.0 (perfect)

### Why Recall Matters Most For Us
In financial distress prediction, missing a distressed company
is much more costly than a false alarm.
A bank that misses a distressed borrower loses money.
A bank that wrongly flags a healthy company just does extra checks.
Therefore we prioritize RECALL over precision.

In [6]:
# Cell 3 — Model 1: Logistic Regression (Baseline)
# Simplest classification model — finds best linear boundary
# between healthy and distressed companies

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, roc_auc_score)

# Initialize the model
# max_iter=1000 → give it enough iterations to converge
# random_state=42 → reproducible results every run
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Storage for cross validation scores
lr_scores = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'roc_auc': []
}

# Run 5-fold cross validation
print("Running 5-Fold Cross Validation — Logistic Regression")
print("="*55)

for fold, (train_idx, test_idx) in enumerate(cv.split(X_scaled, y)):
    # Split data into train and test for this fold
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Train the model on training data
    lr_model.fit(X_train, y_train)
    
    # Predict on test data — data model has never seen
    y_pred = lr_model.predict(X_test)
    y_prob = lr_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics for this fold
    lr_scores['accuracy'].append(accuracy_score(y_test, y_pred))
    lr_scores['precision'].append(precision_score(y_test, y_pred))
    lr_scores['recall'].append(recall_score(y_test, y_pred))
    lr_scores['f1'].append(f1_score(y_test, y_pred))
    lr_scores['roc_auc'].append(roc_auc_score(y_test, y_prob))
    
    print(f"Fold {fold+1}: Accuracy={lr_scores['accuracy'][-1]:.3f} "
          f"Recall={lr_scores['recall'][-1]:.3f} "
          f"ROC AUC={lr_scores['roc_auc'][-1]:.3f}")

# Print final average scores
print("\n" + "="*55)
print("LOGISTIC REGRESSION — FINAL RESULTS")
print("="*55)
for metric, scores in lr_scores.items():
    mean = np.mean(scores)
    std = np.std(scores)
    print(f"{metric:12} → {mean:.3f} ± {std:.3f}")

print(f"\nBaseline (random guessing) → 0.500")
print(f"Our target               → 0.800+")

Running 5-Fold Cross Validation — Logistic Regression
Fold 1: Accuracy=0.917 Recall=0.833 ROC AUC=0.833
Fold 2: Accuracy=0.583 Recall=0.333 ROC AUC=0.611
Fold 3: Accuracy=0.750 Recall=1.000 ROC AUC=0.750
Fold 4: Accuracy=0.583 Recall=0.333 ROC AUC=0.722
Fold 5: Accuracy=0.667 Recall=0.500 ROC AUC=0.722

LOGISTIC REGRESSION — FINAL RESULTS
accuracy     → 0.700 ± 0.125
precision    → 0.750 ± 0.129
recall       → 0.600 ± 0.271
f1           → 0.640 ± 0.188
roc_auc      → 0.728 ± 0.071

Baseline (random guessing) → 0.500
Our target               → 0.800+
